# 18 · Resource-Aware Distributed Tuning

A hyperparameter search explores many configurations, and scheduling every
trial the same way wastes hardware: a lightweight model leaves a whole GPU
mostly idle, a compute-heavy one is slowed down when it has to share one.
WarpRec gives each model the resources that fit its computational profile,
declared in two lines of its configuration. This guide runs the same searches
on one machine with two GPUs, changing only those lines, and measures what
each setting costs and buys. It covers:

- what Ray sees on the machine, and how many trials WarpRec runs at once for
  a given `cpu_per_trial` and `gpu_per_trial`;
- one GPU against two, and two trials packed on each GPU
  (`gpu_per_trial: 0.5`), for a lightweight model (BPR) and a compute-heavy
  one (LightGCN);
- pruning the weaker trials with ASHA, on top of packing;
- one trial trained on two GPUs with DDP (`gpu_per_trial: 2`), for each model;
- the energy and carbon of every run, measured with CodeCarbon.

Guide 10 covers search spaces, strategies and schedulers, guide 11 the swarm
pipeline and the dashboards, guide 15 moving a model from the design pipeline
to Ray.

**Requirements:** a Linux machine with two CUDA GPUs (this notebook was run on
two RTX 4090 and 32 CPU cores) and `pip install "warprec[dashboard]" jupyter`.
Runs in about two hours. MovieLens-100K is small for these GPUs, so LightGCN
is given large embeddings and deep propagation to make it as compute-heavy as
it is on larger data: the numbers show how the settings change a search, not
what the hardware can do.

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from loguru import logger

sys.path.insert(0, "..")
from guide_data import movielens_100k

folder = movielens_100k()  # downloads and converts it as guide 1 shows
RUNS = Path("../runs/18-resource-aware-distributed-tuning")
shutil.rmtree(RUNS, ignore_errors=True)
RUNS.mkdir(parents=True)
logger.disable("warprec")

## 1. The machine and the trials it holds

Ray sees every GPU the process may use. `CUDA_VISIBLE_DEVICES` restricts that,
which is how the runs below compare one GPU with two on the same machine.

In [ ]:
import ray

ray.init(logging_level="ERROR", include_dashboard=False)
node = ray.nodes()[0]["Resources"]
print({key: node[key] for key in ("CPU", "GPU")})

A trial is a Ray Train run: one worker, or one per GPU when `gpu_per_trial` is
an integer above 1, which then trains with DDP. Unless
`max_concurrent_trials` is set, WarpRec runs as many trials at once as the
nodes can host, counting each worker against the node's scarcest resource
(guide 10). Every configuration here gives each worker 4 CPUs, which size its
data loader (3 processes): 4 per trial, 8 for a trial on two GPUs. With 32
cores the GPUs are the scarcer resource. For the footprints this guide uses:

In [ ]:
from warprec.recommenders.trainer import Trainer
from warprec.recommenders.trainer.trainer import trials_that_fit
from warprec.utils.config.config import load_train_configuration

trainer = Trainer(storage_path=str(RUNS / "probe"))
footprints = {}
for name in ["bpr", "bpr-packed", "lightgcn", "lightgcn-ddp"]:
    config = load_train_configuration(f"configs/{name}.yml")
    optimization = next(iter(config.models.values()))["optimization"]
    # The footprint the trainer computes for a trial of this configuration
    scaling = trainer._get_scaling_config(
        optimization["cpu_per_trial"], optimization["gpu_per_trial"], device="cuda"
    )
    footprints[name] = {
        "gpu_per_trial": optimization["gpu_per_trial"],
        "workers per trial": scaling["num_workers"],
        "each worker": scaling["resources_per_worker"],
        "trials at once": trials_that_fit(ray.nodes(), scaling),
    }
ray.shutdown()
pd.DataFrame(footprints).T

## 2. One GPU, two GPUs, two trials per GPU

Each search is an 8-trial grid, as in Table 4 of the WarpRec paper. BPR
scores a user against an item with an embedding lookup: a batch keeps a GPU
busy for a fraction of a millisecond, and most of a trial goes into sampling
negatives and moving batches. LightGCN propagates the embeddings of the whole
user-item graph at every step, here 512 or 1024 dimensions through 3 or 4
layers, which keeps a GPU busy.

In [ ]:
print(Path("configs/bpr.yml").read_text())

The packed configuration differs in one line, and LightGCN's are built the
same way:

In [ ]:
import difflib


def changed(a, b):
    """The lines of configs/b.yml that differ from configs/a.yml."""
    lines = difflib.unified_diff(
        Path(f"configs/{a}.yml").read_text().splitlines(True),
        Path(f"configs/{b}.yml").read_text().splitlines(True),
        f"{a}.yml",
        f"{b}.yml",
        n=0,
    )
    print("".join(line for line in lines if not line.startswith("@@")))


changed("bpr", "bpr-packed")

A helper runs a configuration as `python -m warprec.run` would, on the GPUs it
is given, and reads back what the run wrote: the time report (guide 11), the
test results and the rows CodeCarbon appended to `emissions.csv`.

In [ ]:
def run(name, gpus="0,1", pipeline="train"):
    """Run configs/<name>.yml on the GPUs given; return what it measured."""
    out = RUNS / name
    shutil.rmtree(out, ignore_errors=True)
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": gpus}
    start = time.perf_counter()
    with open(RUNS / f"{name}-gpus{gpus.replace(',', '')}.log", "w") as log:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "warprec.run",
                "-c",
                f"configs/{name}.yml",
                "-p",
                pipeline,
            ],
            env=env,
            stdout=log,
            stderr=subprocess.STDOUT,
            check=True,
        )
    wall = time.perf_counter() - start
    report = pd.read_csv(next((out / "evaluation").glob("Time_Report*")), sep="\t")
    results = pd.read_csv(next((out / "evaluation").glob("Overall_Results*")), sep="\t")
    emissions = pd.read_csv(out / "ray_results" / "emissions.csv")
    summary = {
        "run (s)": wall,
        "search (s)": report["Hyperparameter Exploration Time"].sum(),
        "mean trial (s)": report["Average Trial Time (s)"].mean(),
        "nDCG@10": results.set_index("Model")["nDCG"].to_dict(),
        "kWh": emissions["energy_consumed"].sum(),
        "kg CO2eq": emissions["emissions"].sum(),
    }
    return summary, report


setups = {
    "1 GPU": ("", "0"),
    "2 GPUs": ("", "0,1"),
    "2 GPUs, 2 trials per GPU": ("-packed", "0,1"),
}
packing = {}
for model in ["bpr", "lightgcn"]:
    for setup, (suffix, gpus) in setups.items():
        summary, _ = run(model + suffix, gpus)
        packing[(model, setup)] = summary

table = pd.DataFrame(packing).T
table["nDCG@10"] = [next(iter(v.values())) for v in table["nDCG@10"]]
table["speed-up"] = [
    table.loc[(m, "1 GPU"), "search (s)"] / row["search (s)"]
    for (m, _), row in table.iterrows()
]
table.astype(float).round(
    {
        "run (s)": 0,
        "search (s)": 0,
        "mean trial (s)": 1,
        "nDCG@10": 4,
        "kWh": 4,
        "kg CO2eq": 4,
        "speed-up": 2,
    }
)

`search (s)` is the search's own wall clock, from the time report; `run (s)`
adds starting Ray, preparing the data and evaluating the best model on the
test set. Every setup trains the same 8 configurations with the same seeds, so
the test nDCG@10 of the best one does not depend on where it ran.

## 3. Pruning with ASHA

Packing makes more trials run at once; a scheduler makes them shorter. ASHA
compares the trials at rungs (here after 4, 8 and 16 epochs) and stops the
weaker ones, freeing their GPU share for the next trial.

In [ ]:
changed("bpr-packed", "bpr-asha")

In [ ]:
asha, _ = run("bpr-asha")


def epochs_per_trial(name):
    """How many epochs each trial of a search ran, from its Ray Tune results."""
    trials = (RUNS / name / "ray_results").glob("*__BPR*/*/progress.csv")
    return sorted(len(pd.read_csv(p)) for p in trials)


pd.DataFrame(
    {
        scheduler: {
            "search (s)": round(summary["search (s)"]),
            "epochs per trial": epochs_per_trial(name),
            "nDCG@10": round(summary["nDCG@10"]["BPR"], 4),
            "kWh": round(summary["kWh"], 4),
        }
        for scheduler, name, summary in [
            ("FIFO", "bpr-packed", packing[("bpr", "2 GPUs, 2 trials per GPU")]),
            ("ASHA", "bpr-asha", asha),
        ]
    }
).T

## 4. One trial on two GPUs

Packing and pruning scale a search horizontally, across trials. DDP scales a
single trial vertically: `gpu_per_trial: 2` trains it with two workers, one
per GPU. Each worker takes half of every epoch's batches, and the gradients
are averaged between the GPUs after every step. That synchronisation is the
price, and it pays off when a step computes enough to hide it. Each model
trains one configuration of its grid, on one GPU and then on two:

In [ ]:
changed("lightgcn-single", "lightgcn-ddp")

In [ ]:
ddp = {}
for model in ["bpr", "lightgcn"]:
    single, single_report = run(f"{model}-single", "0")
    double, double_report = run(f"{model}-ddp", "0,1")
    one = single_report["Best Trial Mean Epoch Train Time (s)"].iloc[0]
    two = double_report["Best Trial Mean Epoch Train Time (s)"].iloc[0]
    ddp[model] = {
        "epoch, 1 GPU (s)": one,
        "epoch, 2 GPUs (s)": two,
        "speed-up": one / two,
        "efficiency": one / two / 2,
        "samples/s, 1 GPU": single_report[
            "Best Trial Train Throughput (samples/s)"
        ].iloc[0],
        "samples/s, 2 GPUs": double_report[
            "Best Trial Train Throughput (samples/s)"
        ].iloc[0],
        "run, 1 GPU (s)": single["run (s)"],
        "run, 2 GPUs (s)": double["run (s)"],
        "kWh, 1 GPU": single["kWh"],
        "kWh, 2 GPUs": double["kWh"],
        "nDCG@10, 1 GPU": next(iter(single["nDCG@10"].values())),
        "nDCG@10, 2 GPUs": next(iter(double["nDCG@10"].values())),
    }
pd.DataFrame(ddp).round(4)

## 5. Energy and carbon

Every run above measured itself: with `tracking_mode: machine`, one CodeCarbon
tracker per Ray node covers the whole run, the GPUs, the CPU and the memory,
idle or not. A setup that keeps the GPUs busy for less time therefore costs
less energy for the same search.

In [ ]:
energy = {
    f"{m} · {s}": {"kWh": v["kWh"], "kg CO2eq": v["kg CO2eq"]}
    for (m, s), v in packing.items()
}
energy["bpr · ASHA"] = {"kWh": asha["kWh"], "kg CO2eq": asha["kg CO2eq"]}
pd.DataFrame(energy).T.round(4)

The [Dashboard configuration](https://warprec.readthedocs.io/en/latest/configuration/dashboard/)
page lists what CodeCarbon measures under each tracking mode, and where it
estimates instead.

## Summary

| Key | What it does | Docs |
|---|---|---|
| `optimization.cpu_per_trial`, `gpu_per_trial` | the resources of a trial; a fraction of a GPU packs trials, an integer above 1 trains one trial on that many GPUs with DDP | [Models](https://warprec.readthedocs.io/en/latest/configuration/models/) |
| `optimization.max_concurrent_trials` | caps the trials running at once; unset, WarpRec computes it from the nodes that can host a trial | [Models](https://warprec.readthedocs.io/en/latest/configuration/models/) |
| `optimization.scheduler: asha`, `properties.max_t`, `grace_period`, `reduction_factor` | stop the weaker trials at each rung | [Models](https://warprec.readthedocs.io/en/latest/configuration/models/) |
| `general.device: cuda` | train and evaluate on the GPUs Ray assigns | [General](https://warprec.readthedocs.io/en/latest/configuration/general/) |
| `dashboard.codecarbon` | energy and carbon of every run, in `ray_results/emissions.csv` | [Dashboard](https://warprec.readthedocs.io/en/latest/configuration/dashboard/) |